In [7]:
import requests
from bs4 import BeautifulSoup
import time
import string
import os
import json
import re
from tqdm.notebook import tqdm

# =========================================================================
# CẤU HÌNH THƯ MỤC CHO MEDICAL GRAPH RAG (Đã fix đường dẫn tương đối)
# Vị trí hiện tại: MedicalGraphRAG/src/data_collection
# Cần lùi 2 cấp (../..) để ra thư mục gốc, sau đó vào data
# =========================================================================
CURRENT_DIR = os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(CURRENT_DIR, "../../data"))

RAW_DIR = os.path.join(BASE_DIR, "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")

# Chia nhỏ processed thành 2 domain để dễ quản lý
PROCESSED_DISEASE_DIR = os.path.join(PROCESSED_DIR, "diseases")
PROCESSED_DRUG_DIR = os.path.join(PROCESSED_DIR, "drugs")

# Tạo thư mục tự động nếu chưa có
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DISEASE_DIR, exist_ok=True)
os.makedirs(PROCESSED_DRUG_DIR, exist_ok=True)

print(f"📍 Notebook đang chạy tại: {CURRENT_DIR}")
print(f"📁 Cây thư mục lưu trữ data đã được định vị chính xác tại: {BASE_DIR}")

📍 Notebook đang chạy tại: c:\Users\MSI\Programming\python\BaoCaoThucTap\MedicalGraphRAG\src\data_collection
📁 Cây thư mục lưu trữ data đã được định vị chính xác tại: c:\Users\MSI\Programming\python\BaoCaoThucTap\MedicalGraphRAG\data


In [8]:
def clean_text(text):
    """Hàm làm sạch khoảng trắng thừa trong văn bản"""
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [9]:
def get_all_disease_links(output_txt_path):
    base_url = "https://www.vinmec.com"
    alphabet = list(string.ascii_lowercase)
    all_links = []
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    for letter in alphabet:
        page = 1
        print(f"\n--- Đang cào các bệnh bắt đầu bằng chữ: {letter.upper()} ---")
        
        while True:
            url = f"{base_url}/vie/tra-cuu-benh/{letter}/page_{page}"
            try:
                response = requests.get(url, headers=headers, timeout=10)
                if response.status_code != 200:
                    break
                
                soup = BeautifulSoup(response.text, 'html.parser')
                ul_target = soup.find('ul', class_='list_result_AZ')
                
                if not ul_target or not ul_target.find_all('li'):
                    break
                
                links_found = 0
                for li in ul_target.find_all('li'):
                    a_tag = li.find('a')
                    if a_tag and a_tag.get('href'):
                        href = a_tag.get('href')
                        full_url = base_url + href if href.startswith('/') else href
                        
                        if full_url not in all_links:
                            all_links.append(full_url)
                            links_found += 1
                
                print(f"Trang {page}: Tìm thấy {links_found} link.")
                if links_found == 0:
                    break
                
                page += 1
                time.sleep(1.5)
                
            except Exception as e:
                print(f"Gặp lỗi tại {url}: {e}")
                break

    # Lưu kết quả ra file txt (RAW)
    with open(output_txt_path, "w", encoding="utf-8") as f:
        for link in all_links:
            f.write(link + "\n")
            
    print(f"✅ Đã lưu {len(all_links)} links bệnh lý vào: {output_txt_path}")
    return all_links

def scrape_disease_details(links_file, output_folder):
    if not os.path.exists(links_file):
        print(f"❌ Lỗi: Không tìm thấy file {links_file}.")
        return
        
    with open(links_file, "r", encoding="utf-8") as f:
        urls = [line.strip() for line in f if line.strip()]
        
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    for url in tqdm(urls, desc="Đang cào chi tiết Bệnh lý (Vinmec)"):
        try:
            disease_slug = url.split("/")[-1]
            if not disease_slug: continue
                
            file_path = os.path.join(output_folder, f"{disease_slug}.json")
            if os.path.exists(file_path): continue
                
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code != 200: continue
                
            soup = BeautifulSoup(response.text, 'html.parser')
            content_div = soup.find('div', class_='content_detail_sick')
            if not content_div: continue
                
            disease_data = {"url": url, "disease_name": "", "sections": []}
            items = content_div.find_all('div', class_='item_detial_sick')
            
            for index, item in enumerate(items):
                h2_tag = item.find('h2', class_='title_detail_sick')
                section_title = clean_text(h2_tag.text) if h2_tag else f"Mục_{index+1}"
                
                if index == 0 and h2_tag:
                    disease_data["disease_name"] = section_title
                
                body_div = item.find('div', class_='body')
                section_content = ""
                
                if body_div:
                    paragraphs = [clean_text(elem.text) for elem in body_div.find_all(['p', 'li']) if clean_text(elem.text)]
                    section_content = "\n".join(paragraphs)
                
                if section_content:
                    disease_data["sections"].append({
                        "section_title": section_title,
                        "content": section_content
                    })
            
            with open(file_path, "w", encoding="utf-8") as json_file:
                json.dump(disease_data, json_file, ensure_ascii=False, indent=4)
                
            time.sleep(1)
        except Exception as e:
            continue

In [10]:
def get_phuocan_drug_links(output_txt_path):
    base_link = "https://ykhoaphuocan.vn/thuvien/duoc-thu"
    alphabet = list(string.ascii_uppercase)
    all_drug_links = []
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    for letter in alphabet:
        url = f"{base_link}/{letter}"
        print(f"Đang quét danh mục thuốc chữ cái: {letter}")
        
        try:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code != 200: continue
                
            soup = BeautifulSoup(response.text, 'html.parser')
            table_target = soup.find('table', class_='table zero-configuration')
            
            if not table_target: continue
                
            tbody = table_target.find('tbody')
            if not tbody: continue
                
            links_in_letter = 0
            for a_tag in tbody.find_all('a'):
                href = a_tag.get('href')
                if href:
                    href = href.strip()
                    if href == "#" or not href: continue
                        
                    full_url = href if href.startswith('http') else f"{base_link}/{href}"
                    if full_url not in all_drug_links:
                        all_drug_links.append(full_url)
                        links_in_letter += 1
                        
            print(f"-> Tìm thấy {links_in_letter} thuốc.")
            time.sleep(1.5)
            
        except Exception as e:
            print(f"Lỗi khi xử lý danh mục {letter}: {e}")
            continue

    # Lưu kết quả ra file txt (RAW)
    with open(output_txt_path, "w", encoding="utf-8") as f:
        for link in all_drug_links:
            f.write(link + "\n")
            
    print(f"✅ Đã lưu {len(all_drug_links)} links thuốc vào: {output_txt_path}")
    return all_drug_links

def scrape_drug_details(links_file, output_folder):
    if not os.path.exists(links_file):
        print(f"❌ Lỗi: Không tìm thấy file {links_file}.")
        return
        
    with open(links_file, "r", encoding="utf-8") as f:
        urls = [line.strip() for line in f if line.strip()]
        
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    keywords = [
        "Tên chung quốc tế", "Mã ATC", "Loại thuốc", "Dạng thuốc và hàm lượng",
        "Dược lý và cơ chế tác dụng", "Chỉ định", "Chống chỉ định", "Thận trọng",
        "Tác dụng không mong muốn (ADR)", "Hướng dẫn cách xử lý ADR", "Liều lượng và cách dùng", "Tương tác thuốc",
        "Độ ổn định và bảo quản", "Quá liều và xử trí"
    ]
    
    for url in tqdm(urls, desc="Đang cào chi tiết Thuốc (Phước An)"):
        try:
            drug_slug = url.split("/")[-1]
            if not drug_slug: continue
                
            file_path = os.path.join(output_folder, f"{drug_slug}.json")
            if os.path.exists(file_path): continue  
                
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code != 200: continue
                
            soup = BeautifulSoup(response.text, 'html.parser')
            h1_title = soup.find('h1', id='idTuaDe')
            drug_name = clean_text(h1_title.text) if h1_title else drug_slug
            
            content_div = soup.find('div', id='idnoidung')
            if not content_div: continue
                
            drug_data = {"url": url, "drug_name": drug_name, "sections": []}
            p_elements = content_div.find_all(['p', 'h1', 'h2', 'h3'])
            
            current_section_title = "Thông tin chung"
            current_section_content = []
            
            for p in p_elements:
                full_text = clean_text(p.text)
                if not full_text: continue
                
                bold_tag = p.find(['b', 'strong', 'i'])
                is_key, matched_kw = False, ""
                
                if bold_tag:
                    bold_text = clean_text(bold_tag.text)
                    for kw in keywords:
                        if kw.lower() in bold_text.lower() and len(bold_text) < 60:
                            is_key, matched_kw = True, bold_text
                            break
                
                if not is_key and len(full_text) < 50:
                    for kw in keywords:
                        if kw.lower() in full_text.lower():
                            is_key, matched_kw = True, full_text
                            break
                            
                if is_key:
                    if current_section_content:
                        drug_data["sections"].append({
                            "section_title": current_section_title,
                            "content": "\n".join(current_section_content)
                        })
                        current_section_content = []
                    
                    if ":" in full_text and len(full_text) > len(matched_kw) + 3:
                        parts = full_text.split(":", 1)
                        drug_data["sections"].append({
                            "section_title": clean_text(parts[0]),
                            "content": clean_text(parts[1])
                        })
                        current_section_title = "Thông tin chung"
                    else:
                        current_section_title = matched_kw.rstrip(":")
                else:
                    current_section_content.append(full_text)
                    
            if current_section_content:
                drug_data["sections"].append({
                    "section_title": current_section_title,
                    "content": "\n".join(current_section_content)
                })
                
            with open(file_path, "w", encoding="utf-8") as json_file:
                json.dump(drug_data, json_file, ensure_ascii=False, indent=4)
                
            time.sleep(1)  
        except Exception as e:
            continue

In [ ]:
# Xác định đường dẫn file RAW
VINMEC_LINKS_FILE = os.path.join(RAW_DIR, "vinmec_disease_links.txt")
PHUOCAN_LINKS_FILE = os.path.join(RAW_DIR, "phuocan_drug_links.txt")

print("========== 🚀 BẮT ĐẦU PIPELINE MEDICAL GRAPH RAG ==========\n")
start_time = time.time()

# ---------------------------------------------------------
# STAGE 1: BỆNH LÝ TỪ VINMEC
# ---------------------------------------------------------
print("--- [STAGE 1] XỬ LÝ DỮ LIỆU BỆNH LÝ (VINMEC) ---")
if not os.path.exists(VINMEC_LINKS_FILE):
    print("⏳ Đang cào Links Bệnh lý...")
    get_all_disease_links(VINMEC_LINKS_FILE)
else:
    print(f"✅ Đã tìm thấy file Links Bệnh lý tại {VINMEC_LINKS_FILE}. Bỏ qua bước quét links.")

print("⏳ Đang cào Nội dung chi tiết Bệnh lý...")
scrape_disease_details(links_file=VINMEC_LINKS_FILE, output_folder=PROCESSED_DISEASE_DIR)
print(f"🎉 Hoàn tất cào bệnh lý! Dữ liệu nằm tại: {PROCESSED_DISEASE_DIR}\n")


# ---------------------------------------------------------
# STAGE 2: DƯỢC THƯ TỪ Y KHOA PHƯỚC AN
# ---------------------------------------------------------
print("--- [STAGE 2] XỬ LÝ DỮ LIỆU DƯỢC THƯ (PHƯỚC AN) ---")
if not os.path.exists(PHUOCAN_LINKS_FILE):
    print("⏳ Đang cào Links Thuốc...")
    get_phuocan_drug_links(PHUOCAN_LINKS_FILE)
else:
    print(f"✅ Đã tìm thấy file Links Thuốc tại {PHUOCAN_LINKS_FILE}. Bỏ qua bước quét links.")

print("⏳ Đang cào Nội dung chi tiết Thuốc...")
scrape_drug_details(links_file=PHUOCAN_LINKS_FILE, output_folder=PROCESSED_DRUG_DIR)
print(f"🎉 Hoàn tất cào Dược thư! Dữ liệu nằm tại: {PROCESSED_DRUG_DIR}\n")


print("===========================================================")
print(f"🏁 Hoàn thành toàn bộ quy trình RAG Pipeline!")
print(f"⏱️ Tổng thời gian xử lý: {round((time.time() - start_time) / 60, 2)} phút.")

========== 🚀 BẮT ĐẦU PIPELINE MEDICAL GRAPH RAG ==========

--- [STAGE 1] XỬ LÝ DỮ LIỆU BỆNH LÝ (VINMEC) ---
⏳ Đang cào Links Bệnh lý...

--- Đang cào các bệnh bắt đầu bằng chữ: A ---
Trang 1: Tìm thấy 17 link.

--- Đang cào các bệnh bắt đầu bằng chữ: B ---
Trang 1: Tìm thấy 30 link.
Trang 2: Tìm thấy 7 link.

--- Đang cào các bệnh bắt đầu bằng chữ: C ---
Trang 1: Tìm thấy 28 link.

--- Đang cào các bệnh bắt đầu bằng chữ: D ---
Trang 1: Tìm thấy 12 link.

--- Đang cào các bệnh bắt đầu bằng chữ: E ---

--- Đang cào các bệnh bắt đầu bằng chữ: F ---

--- Đang cào các bệnh bắt đầu bằng chữ: G ---
Trang 1: Tìm thấy 21 link.

--- Đang cào các bệnh bắt đầu bằng chữ: H ---
Trang 1: Tìm thấy 30 link.
Trang 2: Tìm thấy 30 link.
Trang 3: Tìm thấy 30 link.
Trang 4: Tìm thấy 29 link.

--- Đang cào các bệnh bắt đầu bằng chữ: I ---

--- Đang cào các bệnh bắt đầu bằng chữ: J ---
Trang 1: Tìm thấy 1 link.

--- Đang cào các bệnh bắt đầu bằng chữ: K ---
Trang 1: Tìm thấy 5 link.

--- Đang cào các bệnh bắ

Đang cào chi tiết Bệnh lý (Vinmec):   0%|          | 0/678 [00:00<?, ?it/s]

🎉 Hoàn tất cào bệnh lý! Dữ liệu nằm tại: c:\Users\MSI\Programming\python\BaoCaoThucTap\MedicalGraphRAG\data\processed\diseases

--- [STAGE 2] XỬ LÝ DỮ LIỆU DƯỢC THƯ (PHƯỚC AN) ---
⏳ Đang cào Links Thuốc...
Đang quét danh mục thuốc chữ cái: A
-> Tìm thấy 60 thuốc.
Đang quét danh mục thuốc chữ cái: B
-> Tìm thấy 26 thuốc.
Đang quét danh mục thuốc chữ cái: C
-> Tìm thấy 80 thuốc.
Đang quét danh mục thuốc chữ cái: D
-> Tìm thấy 44 thuốc.
Đang quét danh mục thuốc chữ cái: E
-> Tìm thấy 24 thuốc.
Đang quét danh mục thuốc chữ cái: F
-> Tìm thấy 26 thuốc.
Đang quét danh mục thuốc chữ cái: G
-> Tìm thấy 29 thuốc.
Đang quét danh mục thuốc chữ cái: H
-> Tìm thấy 12 thuốc.
Đang quét danh mục thuốc chữ cái: I
-> Tìm thấy 24 thuốc.
Đang quét danh mục thuốc chữ cái: J
-> Tìm thấy 0 thuốc.
Đang quét danh mục thuốc chữ cái: K
-> Tìm thấy 9 thuốc.
Đang quét danh mục thuốc chữ cái: L
-> Tìm thấy 24 thuốc.
Đang quét danh mục thuốc chữ cái: M
-> Tìm thấy 32 thuốc.
Đang quét danh mục thuốc chữ cái: N
-> Tìm

Đang cào chi tiết Thuốc (Phước An):   0%|          | 0/599 [00:00<?, ?it/s]